In [ ]:
# Cell 0 -- v72 (Perch+PretrainedProtoSSM10K+EffNet+GlobalMLP): Install ONNX Runtime for Perch v2 (replaces TensorFlow)
# v48 = v45 + determinism fix (threads=1) + parallel file loading + spatial embeddings
# Perch ONNX is 9x faster than TF SavedModel on CPU.
import os, glob
os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Install onnxruntime from bundled wheel datasets
_whl_candidates = glob.glob("/kaggle/input/**/onnxruntime*cp312*x86_64*.whl", recursive=True)
if _whl_candidates:
    # Sort to get latest version
    _whl_candidates.sort(reverse=True)
    print(f"Installing onnxruntime from: {_whl_candidates[0]}")
    os.system(f"pip install -q --no-deps '{_whl_candidates[0]}'")
else:
    print("ERROR: No onnxruntime wheel found in mounted datasets!")
    # List all input dirs for debugging
    import subprocess
    subprocess.run(["find", "/kaggle/input", "-maxdepth", "3", "-name", "*.whl"], capture_output=False)

In [ ]:
# Cell 1 -- Mode switch
MODE = "submit"

assert MODE in {"train", "submit"}
print("MODE =", MODE)


In [ ]:
# Cell 2 -- v48: Imports and run config
# v48 = v45 base + determinism fix + parallel loading + spatial embeddings
# Changes from v45: intra_op_num_threads=1 (fix 0.004 variance),
# ThreadPoolExecutor for parallel file loading, spatial_embedding extraction
from concurrent.futures import ThreadPoolExecutor

import gc
import json
import re
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import onnxruntime as ort
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

_WALL_START = time.time()

BASE = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")  # For labels.csv mapping
# v40: ONNX model path (search multiple mount points)
_onnx_candidates = [
    Path("/kaggle/input/perch-onnx/perch_v2.onnx"),
    Path("/kaggle/input/datasets/mlnjsh/perch-onnx/perch_v2.onnx"),
]
ONNX_MODEL_PATH = next((p for p in _onnx_candidates if p.exists()), _onnx_candidates[0])
print(f"ONNX model path: {ONNX_MODEL_PATH} (exists={ONNX_MODEL_PATH.exists()})")

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES = 60 * SR
N_WINDOWS = 12
DEVICE = torch.device("cpu")  # Competition: CPU only
LOGS = {}

CFG = {
    "mode": MODE,
    "verbose": MODE == "train",
    "run_oof_baseline": False,
    "run_probe_check": False,
    "run_probe_grid": False,
    # v28: batch_files=32 in submit mode saves ~1 min vs v20's 16
    "batch_files": 16 if MODE == "train" else 4,  # v40: smaller batch for ONNX (less RAM than TF)
    "proxy_reduce": "max",
    "dryrun_n_files": 50 if MODE == "train" else 20,
    "require_full_cache_in_submit": False,
    "full_cache_input_dir": Path("/kaggle/input/datasets/jaejohn/perch-meta"),
    "full_cache_work_dir": Path("/kaggle/working/perch_cache"),
    # v28: Per-taxon prior lambdas (from pantanal-distill)
    "best_fusion": {
        "lambda_event": 0.4,
        "lambda_texture": 1.0,
        "lambda_proxy_texture": 0.8,
        "smooth_texture": 0.35,  # weighted average for Amphibia/Insecta
        "smooth_event": 0.15,    # soft max-pool for Aves/Mammalia/Reptilia
    },
    # v28: SACRED v20 probe params (PCA=48, C=2.0) as LogReg fallback
    "frozen_best_probe": {
        "pca_dim": 48,
        "min_pos": 6,
        "C": 2.0,
        "alpha": 0.25,
    },
    # v28: MLP params (from pantanal-distill, replaces LogReg as primary)
    "probe_backend": "mlp",
    "mlp_params": {
        "hidden_layer_sizes": (128,),
        "activation": "relu",
        "max_iter": 200,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 15,
        "random_state": 42,
        "learning_rate_init": 0.001,
        "alpha": 0.01,
    },
    # v28: ProtoSSM v2 architecture (from pantanal-distill)
    "proto_ssm": {
        "d_model": 128,
        "d_state": 16,
        "n_ssm_layers": 2,
        "dropout": 0.15,
        "n_sites": 20,
        "meta_dim": 16,
    },
    # v28: ProtoSSM training config
    "proto_ssm_train": {
        "n_epochs": 80,
        "lr": 1e-3,
        "weight_decay": 2e-3,
        "patience": 15,
        "pos_weight_cap": 30.0,
        "distill_weight": 0.1,
        "label_smoothing": 0.02,
        "oof_n_splits": 3,
    },
    # v28: ResidualSSM config (from pantanal-distill)
    "residual_ssm": {
        "d_model": 64,
        "d_state": 8,
        "dropout": 0.1,
        "correction_weight": 0.0,
        "n_epochs": 20,
        "lr": 1e-3,
        "patience": 8,
    },
    # v28: Temperature scaling (per-taxon)
    "temperature": {
        "aves": 1.10,
        "texture": 0.95,
    },
    # v28: File-level confidence scaling
    "file_level_top_k": 2,
    # v61: Global MLP weights (pre-trained on train_audio Perch embeddings)
    "global_mlp_weight": 0.06,  # Weight in final blend (0.06 = 6%)
    "global_mlp_path": Path("/kaggle/input/perch-mlp-weights/perch_mlp_weights.npz"),
}

CFG["full_cache_work_dir"].mkdir(parents=True, exist_ok=True)

print("ONNX Runtime:", ort.__version__)
print("PyTorch:", torch.__version__)
print("Competition dir exists:", BASE.exists())
print("Model dir exists:", MODEL_DIR.exists())


In [ ]:
# Cell 3 -- Data loading and preprocessing (identical to v20)
taxonomy = pd.read_csv(BASE / "taxonomy.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)

taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
soundscape_labels["primary_label"] = soundscape_labels["primary_label"].astype(str)

def parse_soundscape_labels(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(";") if t.strip()]

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {"file_id": None, "site": None, "date": pd.NaT, "time_utc": None, "hour_utc": -1, "month": -1}
    file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")
    return {"file_id": file_id, "site": site, "date": dt, "time_utc": hms, "hour_utc": int(hms[:2]), "month": int(dt.month) if pd.notna(dt) else -1}

def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_soundscape_labels(x)))

sc_clean = (
    soundscape_labels
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)

sc_clean["start_sec"] = pd.to_timedelta(sc_clean["start"]).dt.total_seconds().astype(int)
sc_clean["end_sec"] = pd.to_timedelta(sc_clean["end"]).dt.total_seconds().astype(int)
sc_clean["row_id"] = sc_clean["filename"].str.replace(".ogg", "", regex=False) + "_" + sc_clean["end_sec"].astype(str)

meta = sc_clean["filename"].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta], axis=1)

windows_per_file = sc_clean.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc_clean["file_fully_labeled"] = sc_clean["filename"].isin(full_files)

label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}
Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean["label_list"]):
    idxs = [label_to_idx[lbl] for lbl in labels if lbl in label_to_idx]
    if idxs:
        Y_SC[i, idxs] = 1

full_truth = sc_clean[sc_clean["file_fully_labeled"]].sort_values(["filename", "end_sec"]).reset_index(drop=False)
Y_FULL_TRUTH = Y_SC[full_truth["index"].to_numpy()]

print("sc_clean:", sc_clean.shape)
print("Y_SC:", Y_SC.shape)
print("Full files:", len(full_files))
print("Trusted full windows:", len(full_truth))


In [ ]:
# Cell 4 -- v48: Load Perch v2 ONNX session + mapping + genus proxies
BEST = CFG["best_fusion"]

# v40: ONNX session replaces TF SavedModel (9x faster)
_onnx_t0 = time.time()
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_options.intra_op_num_threads = 1  # v48: single thread for deterministic results (v45 had 0.004 variance with threads=2)
sess_options.inter_op_num_threads = 1
sess_options.enable_mem_pattern = False  # v40: reduce memory allocation
sess_options.enable_cpu_mem_arena = False  # v40: don't pre-allocate memory pool
ort_session = ort.InferenceSession(str(ONNX_MODEL_PATH), sess_options)
print(f"v40: ONNX session loaded in {time.time()-_onnx_t0:.1f}s")

bc_labels = pd.read_csv(MODEL_DIR / "assets" / "labels.csv").reset_index().rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
NO_LABEL_INDEX = len(bc_labels)
MANUAL_SCIENTIFIC_NAME_MAP = {}
taxonomy = taxonomy.copy()
taxonomy["scientific_name_lookup"] = taxonomy["scientific_name"].replace(MANUAL_SCIENTIFIC_NAME_MAP)
bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})
mapping = taxonomy.merge(bc_lookup[["scientific_name_lookup", "bc_index"]], on="scientific_name_lookup", how="left")
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)
label_to_bc_index = mapping.set_index("primary_label")["bc_index"]
BC_INDICES = np.array([int(label_to_bc_index.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA = {"Amphibia", "Insecta"}
ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]
idx_active_texture = np.array([label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA], dtype=np.int32)
idx_active_event = np.array([label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA], dtype=np.int32)

idx_mapped_active_texture = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive = np.array([i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES], dtype=np.int32)

# Genus proxies for unmapped species
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)].copy()

def get_genus_hits(scientific_name):
    genus = str(scientific_name).split()[0]
    hits = bc_labels[bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)].copy()
    return genus, hits

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    target = row["primary_label"]
    sci = row["scientific_name"]
    genus, hits = get_genus_hits(sci)
    if len(hits) > 0:
        proxy_map[target] = {"bc_indices": hits["bc_index"].astype(int).tolist()}

PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
SELECTED_PROXY_TARGETS = sorted([t for t in proxy_map.keys() if CLASS_NAME_MAP.get(t) in PROXY_TAXA])
selected_proxy_pos = np.array([label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32)
selected_proxy_pos_to_bc = {label_to_idx[target]: np.array(proxy_map[target]["bc_indices"], dtype=np.int32) for target in SELECTED_PROXY_TARGETS}
idx_selected_proxy_active_texture = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)

print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES}")
print(f"Proxy targets: {len(SELECTED_PROXY_TARGETS)}")


In [ ]:
# Cell 5 -- Metrics and helper utilities
# v28: Per-taxon smoothing (from pantanal-distill)

def macro_auc_skip_empty(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

# Texture taxa (Amphibia, Insecta): weighted average with neighbors
def smooth_cols_fixed12(scores, cols, alpha=0.35):
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    assert len(s) % N_WINDOWS == 0
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return s

# v28 NEW: Event taxa (Aves, Mammalia, Reptilia): soft max-pool with neighbors
def smooth_events_fixed12(scores, cols, alpha=0.15):
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    assert len(s) % N_WINDOWS == 0
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    local_max = np.maximum(x, np.maximum(prev_x, next_x))
    view[:, :, cols] = (1.0 - alpha) * x + alpha * local_max
    return s

def seq_features_1d(v):
    assert len(v) % N_WINDOWS == 0
    x = v.reshape(-1, N_WINDOWS)
    prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean_v = np.repeat(x.mean(axis=1), N_WINDOWS)
    max_v = np.repeat(x.max(axis=1), N_WINDOWS)
    std_v = np.repeat(x.std(axis=1), N_WINDOWS)
    return prev_v, next_v, mean_v, max_v, std_v

# v28: File-level confidence scaling (from timeoptimized notebook)
def file_level_confidence_scale(preds, n_windows=12, top_k=2):
    N, C = preds.shape
    assert N % n_windows == 0
    view = preds.reshape(-1, n_windows, C)
    sorted_view = np.sort(view, axis=1)
    top_k_mean = sorted_view[:, -top_k:, :].mean(axis=1, keepdims=True)
    scaled = view * top_k_mean
    return scaled.reshape(N, C)

print("Utilities defined (per-taxon smoothing, file-level confidence scaling)")


In [ ]:
# Cell 6 -- v48: Perch ONNX inference with embeddings + spatial + parallel loading
def read_soundscape_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        raise ValueError(f"Unexpected sample rate {sr}")
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES:
        y = y[:FILE_SAMPLES]
    return y

def infer_perch_with_embeddings(paths, batch_files=16, verbose=True, proxy_reduce="max"):
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * N_WINDOWS
    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)
    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536), dtype=np.float32)
    write_row = 0
    iterator = range(0, n_files, batch_files)
    if verbose:
        iterator = tqdm(iterator, total=(n_files + batch_files - 1) // batch_files, desc="Perch batches")
    for start in iterator:
        batch_paths = paths[start:start + batch_files]
        batch_n = len(batch_paths)
        x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
        batch_row_start = write_row
        x_pos = 0
        # v48: parallel file loading with ThreadPoolExecutor
        def _load_one(path):
            return read_soundscape_60s(path)
        with ThreadPoolExecutor(max_workers=min(4, batch_n)) as pool:
            batch_audio = list(pool.map(_load_one, batch_paths))
        for j, path in enumerate(batch_paths):
            y = batch_audio[j]
            x[x_pos:x_pos + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
            meta_info = parse_soundscape_filename(path.name)
            stem = path.stem
            row_ids[write_row:write_row + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
            filenames[write_row:write_row + N_WINDOWS] = path.name
            sites[write_row:write_row + N_WINDOWS] = meta_info["site"]
            hours[write_row:write_row + N_WINDOWS] = int(meta_info["hour_utc"])
            x_pos += N_WINDOWS
            write_row += N_WINDOWS
        # v48: ONNX inference -- request embedding + label + spatial_embedding
        emb, logits, spatial_emb = ort_session.run(
            ["embedding", "label", "spatial_embedding"], {"inputs": x.astype(np.float32)})
        emb = emb.astype(np.float32, copy=False)
        logits = logits.astype(np.float32, copy=False)
        # v48: spatial_embedding shape (B, 16, 4, 1536) -- save mean over spatial dim
        # spatial_emb_temporal = spatial_emb.mean(axis=2)  # (B, 16, 1536) -- for future LSE experiments
        del spatial_emb  # free memory (not used in probe pipeline yet)
        scores[batch_row_start:write_row, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]
        embeddings[batch_row_start:write_row] = emb
        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            sub = logits[:, bc_idx_arr]
            proxy_score = sub.max(axis=1) if proxy_reduce == "max" else sub.mean(axis=1)
            scores[batch_row_start:write_row, pos] = proxy_score.astype(np.float32)
        del x, logits, emb
        gc.collect()
        if hasattr(ort_session, "_sess"): pass  # keep session alive
    meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames, "site": sites, "hour_utc": hours})
    return meta_df, scores, embeddings


In [ ]:
# Cell 7 -- Load or compute full-file Perch cache
def resolve_full_cache_paths():
    candidates = [
        (CFG["full_cache_work_dir"] / "full_perch_meta.parquet", CFG["full_cache_work_dir"] / "full_perch_arrays.npz"),
        (Path("/kaggle/working/full_perch_meta.parquet"), Path("/kaggle/working/full_perch_arrays.npz")),
    ]
    if CFG["full_cache_input_dir"].exists():
        candidates.append((CFG["full_cache_input_dir"] / "full_perch_meta.parquet", CFG["full_cache_input_dir"] / "full_perch_arrays.npz"))
    for meta_path, npz_path in candidates:
        if meta_path.exists() and npz_path.exists():
            return meta_path, npz_path
    return None, None

cache_meta, cache_npz = resolve_full_cache_paths()
if cache_meta is not None:
    print("Loading cached Perch outputs:", cache_meta)
    meta_full = pd.read_parquet(cache_meta)
    arr = np.load(cache_npz)
    scores_full_raw = arr["scores_full_raw"].astype(np.float32)
    emb_full = arr["emb_full"].astype(np.float32)
else:
    print("No cache found. Running Perch on trusted full files...")
    full_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
    meta_full, scores_full_raw, emb_full = infer_perch_with_embeddings(full_paths, batch_files=CFG["batch_files"], verbose=True, proxy_reduce=CFG["proxy_reduce"])
    meta_full.to_parquet(CFG["full_cache_work_dir"] / "full_perch_meta.parquet", index=False)
    np.savez_compressed(CFG["full_cache_work_dir"] / "full_perch_arrays.npz", scores_full_raw=scores_full_raw, emb_full=emb_full)

full_truth_aligned = full_truth.set_index("row_id").loc[meta_full["row_id"]].reset_index()
Y_FULL = Y_SC[full_truth_aligned["index"].to_numpy()]
print("meta_full:", meta_full.shape, "emb_full:", emb_full.shape)


In [ ]:
# Cell 8 -- Fold-safe metadata prior tables (v28: per-taxon smoothing)
def fit_prior_tables(prior_df, Y_prior):
    prior_df = prior_df.reset_index(drop=True)
    global_p = Y_prior.mean(axis=0).astype(np.float32)
    site_keys = sorted(prior_df["site"].dropna().astype(str).unique().tolist())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), Y_prior.shape[1]), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]
        mask = prior_df["site"].astype(str).values == s
        site_n[i] = mask.sum()
        site_p[i] = Y_prior[mask].mean(axis=0)
    hour_keys = sorted(prior_df["hour_utc"].dropna().astype(int).unique().tolist())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), Y_prior.shape[1]), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]
        mask = prior_df["hour_utc"].astype(int).values == h
        hour_n[i] = mask.sum()
        hour_p[i] = Y_prior[mask].mean(axis=0)
    sh_to_i, sh_n_list, sh_p_list = {}, [], []
    for (s, h), idx in prior_df.groupby(["site", "hour_utc"]).groups.items():
        sh_to_i[(str(s), int(h))] = len(sh_n_list)
        idx = np.array(list(idx))
        sh_n_list.append(len(idx))
        sh_p_list.append(Y_prior[idx].mean(axis=0))
    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if sh_p_list else np.zeros((0, Y_prior.shape[1]), dtype=np.float32)
    return {"global_p": global_p, "site_to_i": site_to_i, "site_n": site_n, "site_p": site_p, "hour_to_i": hour_to_i, "hour_n": hour_n, "hour_p": hour_p, "sh_to_i": sh_to_i, "sh_n": sh_n, "sh_p": sh_p}

def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n = len(sites)
    p = np.repeat(tables["global_p"][None, :], n, axis=0).astype(np.float32, copy=True)
    site_idx = np.fromiter((tables["site_to_i"].get(str(s), -1) for s in sites), dtype=np.int32, count=n)
    hour_idx = np.fromiter((tables["hour_to_i"].get(int(h), -1) if int(h) >= 0 else -1 for h in hours), dtype=np.int32, count=n)
    sh_idx = np.fromiter((tables["sh_to_i"].get((str(s), int(h)), -1) if int(h) >= 0 else -1 for s, h in zip(sites, hours)), dtype=np.int32, count=n)
    valid = hour_idx >= 0
    if valid.any():
        nh = tables["hour_n"][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables["hour_p"][hour_idx[valid]] + (1.0 - wh) * p[valid]
    valid = site_idx >= 0
    if valid.any():
        ns = tables["site_n"][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables["site_p"][site_idx[valid]] + (1.0 - ws) * p[valid]
    valid = sh_idx >= 0
    if valid.any():
        nsh = tables["sh_n"][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables["sh_p"][sh_idx[valid]] + (1.0 - wsh) * p[valid]
    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)

# v28: fuse with BOTH texture and event smoothing (pantanal-distill style)
def fuse_scores_with_tables(base_scores, sites, hours, tables,
                            lambda_event=BEST["lambda_event"],
                            lambda_texture=BEST["lambda_texture"],
                            lambda_proxy_texture=BEST["lambda_proxy_texture"],
                            smooth_texture=BEST["smooth_texture"],
                            smooth_event=BEST["smooth_event"]):
    scores = base_scores.copy()
    prior = prior_logits_from_tables(sites, hours, tables)
    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += lambda_event * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += lambda_texture * prior[:, idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += lambda_proxy_texture * prior[:, idx_selected_proxy_active_texture]
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = lambda_event * prior[:, idx_selected_prioronly_active_event]
    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = lambda_texture * prior[:, idx_selected_prioronly_active_texture]
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0
    # v28: per-taxon smoothing
    scores = smooth_cols_fixed12(scores, idx_active_texture, alpha=smooth_texture)
    scores = smooth_events_fixed12(scores, idx_active_event, alpha=smooth_event)
    return scores.astype(np.float32, copy=False), prior

print("Prior tables and per-taxon smoothing defined.")


In [ ]:
# Cell 9 -- OOF base/prior meta-features (from v20)
def build_oof_base_prior(scores_full_raw, meta_full, sc_clean, Y_SC, n_splits=5, verbose=True):
    groups_full = meta_full["filename"].to_numpy()
    gkf = GroupKFold(n_splits=n_splits)
    oof_base = np.zeros_like(scores_full_raw, dtype=np.float32)
    oof_prior = np.zeros_like(scores_full_raw, dtype=np.float32)
    fold_id = np.full(len(meta_full), -1, dtype=np.int16)
    splits = list(gkf.split(scores_full_raw, groups=groups_full))
    for fold, (tr_idx, va_idx) in enumerate(splits, 1):
        tr_idx, va_idx = np.sort(tr_idx), np.sort(va_idx)
        val_files = set(meta_full.iloc[va_idx]["filename"].tolist())
        prior_mask = ~sc_clean["filename"].isin(val_files).values
        tables = fit_prior_tables(sc_clean.loc[prior_mask].reset_index(drop=True), Y_SC[prior_mask])
        va_base, va_prior = fuse_scores_with_tables(scores_full_raw[va_idx], sites=meta_full.iloc[va_idx]["site"].to_numpy(), hours=meta_full.iloc[va_idx]["hour_utc"].to_numpy(), tables=tables)
        oof_base[va_idx] = va_base
        oof_prior[va_idx] = va_prior
        fold_id[va_idx] = fold
    return oof_base, oof_prior, fold_id

print("Building honest 5-fold OOF meta-features...")
oof_base, oof_prior, oof_fold_id = build_oof_base_prior(scores_full_raw, meta_full, sc_clean, Y_SC, n_splits=5, verbose=False)
baseline_oof_auc = macro_auc_skip_empty(Y_FULL, oof_base)
print(f"OOF baseline AUC: {baseline_oof_auc:.6f}")


In [ ]:
# Cell 10 -- Classwise embedding-probe helpers (v28: pantanal-distill features)
def build_class_features(emb_proj, raw_col, prior_col, base_col):
    prev_base, next_base, mean_base, max_base, std_base = seq_features_1d(base_col)
    diff_mean = base_col - mean_base
    diff_prev = base_col - prev_base
    diff_next = base_col - next_base
    feats = np.concatenate([
        emb_proj,
        raw_col[:, None],
        prior_col[:, None],
        base_col[:, None],
        prev_base[:, None],
        next_base[:, None],
        mean_base[:, None],
        max_base[:, None],
        std_base[:, None],
        diff_mean[:, None],
        diff_prev[:, None],
        diff_next[:, None],
        (raw_col * prior_col)[:, None],
        (raw_col * base_col)[:, None],
        (prior_col * base_col)[:, None],
    ], axis=1)
    return feats.astype(np.float32, copy=False)

print("build_class_features defined")


In [ ]:
# Cell 11 -- Fit final prior tables, PCA, and MLP probes
BEST_PROBE = CFG["frozen_best_probe"]
final_prior_tables = fit_prior_tables(sc_clean.reset_index(drop=True), Y_SC)

# PCA on embeddings
emb_scaler = StandardScaler()
emb_full_scaled = emb_scaler.fit_transform(emb_full)
n_comp = min(int(BEST_PROBE["pca_dim"]), emb_full_scaled.shape[0] - 1, emb_full_scaled.shape[1])
emb_pca = PCA(n_components=n_comp)
Z_FULL = emb_pca.fit_transform(emb_full_scaled).astype(np.float32)
print(f"PCA: {emb_full.shape} -> {Z_FULL.shape}, explained var: {emb_pca.explained_variance_ratio_.sum():.4f}")

# v28: Train MLP(128) probes per class (from pantanal-distill, replaces LogReg)
full_pos_counts = Y_FULL.sum(axis=0)
PROBE_CLASS_IDX = np.where(full_pos_counts >= int(BEST_PROBE["min_pos"]))[0].astype(np.int32)

probe_models = {}
for cls_idx in PROBE_CLASS_IDX:
    y = Y_FULL[:, cls_idx]
    if y.sum() == 0 or y.sum() == len(y):
        continue
    X_cls = build_class_features(Z_FULL, raw_col=scores_full_raw[:, cls_idx], prior_col=oof_prior[:, cls_idx], base_col=oof_base[:, cls_idx])
    n_pos = int(y.sum())
    n_neg = len(y) - n_pos
    # Oversampling for class balance (MLP does not support sample_weight)
    if n_pos > 0 and n_neg > n_pos:
        repeat = max(1, n_neg // n_pos)
        pos_idx = np.where(y == 1)[0]
        X_bal = np.vstack([X_cls, np.tile(X_cls[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(len(pos_idx) * repeat, dtype=y.dtype)])
    else:
        X_bal, y_bal = X_cls, y
    clf = MLPClassifier(**CFG["mlp_params"])
    clf.fit(X_bal, y_bal)
    probe_models[cls_idx] = clf

print(f"MLP probes trained: {len(probe_models)}")
print(f"Wall time: {(time.time() - _WALL_START) / 60:.1f} min")


In [ ]:
# Cell 12 -- ProtoSSM v2: Selective State Space Model (from pantanal-distill)

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv - 1, groups=d_model)
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_size, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        B_t = self.B_proj(x_conv)
        C_t = self.C_proj(x_conv)
        A = -torch.exp(self.A_log)
        y = self._selective_scan(x_conv, dt, A, B_t, C_t)
        y = y * F.silu(z)
        return self.out_proj(y)

    def _selective_scan(self, x, dt, A, B, C):
        batch, T, D = x.shape
        N = self.d_state
        h = torch.zeros(batch, D, N, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(T):
            dt_t = dt[:, t, :, None]
            dA = torch.exp(A[None] * dt_t)
            dB = dt_t * B[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            y_t = (h * C[:, t, None, :]).sum(-1)
            ys.append(y_t)
        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]


class ProtoSSMv2(nn.Module):
    def __init__(self, d_input=1536, d_model=128, d_state=16, n_ssm_layers=2,
                 n_classes=234, n_windows=12, dropout=0.15, n_sites=20, meta_dim=16):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.n_windows = n_windows
        self.input_proj = nn.Sequential(nn.Linear(d_input, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.ssm_fwd = nn.ModuleList()
        self.ssm_bwd = nn.ModuleList()
        self.ssm_merge = nn.ModuleList()
        self.ssm_norm = nn.ModuleList()
        for _ in range(n_ssm_layers):
            self.ssm_fwd.append(SelectiveSSM(d_model, d_state))
            self.ssm_bwd.append(SelectiveSSM(d_model, d_state))
            self.ssm_merge.append(nn.Linear(2 * d_model, d_model))
            self.ssm_norm.append(nn.LayerNorm(d_model))
        self.ssm_drop = nn.Dropout(dropout)
        self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp = nn.Parameter(torch.tensor(5.0))
        self.class_bias = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))
        self.n_families = 0
        self.family_head = None

    def init_prototypes_from_data(self, embeddings, labels):
        with torch.no_grad():
            h = self.input_proj(embeddings)
            for c in range(self.n_classes):
                mask = labels[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def init_family_head(self, n_families, class_to_family):
        self.n_families = n_families
        self.family_head = nn.Linear(self.d_model, n_families)
        self.register_buffer("class_to_family", torch.tensor(class_to_family, dtype=torch.long))

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            s_emb = self.site_emb(site_ids)
            h_emb = self.hour_emb(hours)
            meta = self.meta_proj(torch.cat([s_emb, h_emb], dim=-1))
            h = h + meta[:, None, :]
        for fwd, bwd, merge, norm in zip(self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm):
            residual = h
            h_f = fwd(h)
            h_b = bwd(h.flip(1)).flip(1)
            h = merge(torch.cat([h_f, h_b], dim=-1))
            h = self.ssm_drop(h)
            h = norm(h + residual)
        h_norm = F.normalize(h, dim=-1)
        p_norm = F.normalize(self.prototypes, dim=-1)
        temp = F.softplus(self.proto_temp)
        sim = torch.matmul(h_norm, p_norm.T) * temp + self.class_bias[None, None, :]
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            species_logits = alpha * sim + (1 - alpha) * perch_logits
        else:
            species_logits = sim
        family_logits = None
        if self.family_head is not None:
            family_logits = self.family_head(h.mean(dim=1))
        return species_logits, family_logits, h

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

print("ProtoSSM v2 architecture defined.")


In [ ]:
# Cell 13 -- ProtoSSM v2 Training + ResidualSSM helpers

def build_taxonomy_groups(taxonomy_df, primary_labels):
    for col in ["family", "order", "class_name"]:
        if col in taxonomy_df.columns:
            group_map = taxonomy_df.set_index("primary_label")[col].to_dict()
            break
    else:
        group_map = {label: "Unknown" for label in primary_labels}
    groups = sorted(set(group_map.values()))
    grp_to_idx = {g: i for i, g in enumerate(groups)}
    class_to_group = [grp_to_idx.get(group_map.get(label, "Unknown"), 0) for label in primary_labels]
    return len(groups), class_to_group, grp_to_idx

def build_site_mapping(meta_df):
    sites = meta_df["site"].unique().tolist()
    site_to_idx = {s: i + 1 for i, s in enumerate(sites)}
    return site_to_idx, len(sites) + 1

def reshape_to_files(flat_array, meta_df, n_windows=N_WINDOWS):
    filenames = meta_df["filename"].to_numpy()
    unique_files = []
    seen = set()
    for f in filenames:
        if f not in seen:
            unique_files.append(f)
            seen.add(f)
    n_files = len(unique_files)
    new_shape = (n_files, n_windows) + flat_array.shape[1:]
    return flat_array.reshape(new_shape), unique_files

def get_file_metadata(meta_df, file_list, site_to_idx, n_sites_max):
    file_to_row = {}
    filenames = meta_df["filename"].to_numpy()
    sites_arr = meta_df["site"].to_numpy()
    hours_arr = meta_df["hour_utc"].to_numpy()
    for i, f in enumerate(filenames):
        if f not in file_to_row:
            file_to_row[f] = i
    site_ids = np.zeros(len(file_list), dtype=np.int64)
    hour_ids = np.zeros(len(file_list), dtype=np.int64)
    for fi, fname in enumerate(file_list):
        row = file_to_row.get(fname)
        if row is not None:
            site_ids[fi] = min(site_to_idx.get(sites_arr[row], 0), n_sites_max - 1)
            hour_ids[fi] = int(hours_arr[row]) % 24
    return site_ids, hour_ids

def train_proto_ssm_single(model, emb_train, logits_train, labels_train,
                           site_ids_train=None, hours_train=None,
                           emb_val=None, logits_val=None, labels_val=None,
                           site_ids_val=None, hours_val=None,
                           file_families_train=None, file_families_val=None,
                           cfg=None, verbose=True):
    if cfg is None:
        cfg = CFG["proto_ssm_train"]
    label_smoothing = cfg.get("label_smoothing", 0.0)
    emb_tr = torch.tensor(emb_train, dtype=torch.float32)
    logits_tr = torch.tensor(logits_train, dtype=torch.float32)
    labels_tr = torch.tensor(labels_train, dtype=torch.float32)
    if label_smoothing > 0:
        labels_tr = labels_tr * (1.0 - label_smoothing) + label_smoothing / 2.0
    site_tr = torch.tensor(site_ids_train, dtype=torch.long) if site_ids_train is not None else None
    hour_tr = torch.tensor(hours_train, dtype=torch.long) if hours_train is not None else None
    has_val = emb_val is not None
    if has_val:
        emb_v = torch.tensor(emb_val, dtype=torch.float32)
        logits_v = torch.tensor(logits_val, dtype=torch.float32)
        labels_v = torch.tensor(labels_val, dtype=torch.float32)
        site_v = torch.tensor(site_ids_val, dtype=torch.long) if site_ids_val is not None else None
        hour_v = torch.tensor(hours_val, dtype=torch.long) if hours_val is not None else None
    fam_tr = torch.tensor(file_families_train, dtype=torch.float32) if file_families_train is not None else None
    pos_counts = labels_tr.sum(dim=(0, 1))
    total = labels_tr.shape[0] * labels_tr.shape[1]
    pos_weight = ((total - pos_counts) / (pos_counts + 1)).clamp(max=cfg["pos_weight_cap"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=cfg["lr"], epochs=cfg["n_epochs"], steps_per_epoch=1, pct_start=0.1, anneal_strategy="cos")
    best_val_loss = float("inf")
    best_state = None
    wait = 0
    for epoch in range(cfg["n_epochs"]):
        model.train()
        species_out, family_out, _ = model(emb_tr, logits_tr, site_ids=site_tr, hours=hour_tr)
        loss_bce = F.binary_cross_entropy_with_logits(species_out, labels_tr, pos_weight=pos_weight[None, None, :])
        loss_distill = F.mse_loss(species_out, logits_tr)
        loss = loss_bce + cfg["distill_weight"] * loss_distill
        if family_out is not None and fam_tr is not None:
            loss = loss + 0.1 * F.binary_cross_entropy_with_logits(family_out, fam_tr)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        model.eval()
        with torch.no_grad():
            if has_val:
                val_out, _, _ = model(emb_v, logits_v, site_ids=site_v, hours=hour_v)
                val_loss = F.binary_cross_entropy_with_logits(val_out, labels_v, pos_weight=pos_weight[None, None, :])
            else:
                val_loss = loss
        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if verbose and (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}: train={loss.item():.4f} val={val_loss.item():.4f} wait={wait}")
        if wait >= cfg["patience"]:
            if verbose:
                print(f"  Early stop at epoch {epoch+1}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, {"best_val_loss": best_val_loss}


def optimize_ensemble_weight(oof_proto_flat, oof_mlp_flat, y_true_flat):
    weights = np.arange(0.0, 1.05, 0.05)
    results = []
    for w in weights:
        blended = w * oof_proto_flat + (1.0 - w) * oof_mlp_flat
        try:
            auc = macro_auc_skip_empty(y_true_flat, blended)
        except Exception:
            auc = 0.0
        results.append((w, auc))
    best_w, best_auc = max(results, key=lambda x: x[1])
    return best_w, best_auc, results

print("ProtoSSM training and ensemble weight optimizer defined.")


In [ ]:
# Cell 14 -- v72: Load pretrained ProtoSSM (10K files) + fine-tune on labeled data
_wall_before_proto = (time.time() - _WALL_START) / 60.0
print(f"Wall time before ProtoSSM: {_wall_before_proto:.1f} min")

# Reshape to file-level
emb_files, file_list = reshape_to_files(emb_full, meta_full)
logits_files, _ = reshape_to_files(scores_full_raw, meta_full)
labels_files, _ = reshape_to_files(Y_FULL, meta_full)
print(f"File-level shapes: emb={emb_files.shape}, logits={logits_files.shape}")

# Taxonomy groups and site mapping
n_families, class_to_family, fam_to_idx = build_taxonomy_groups(taxonomy, PRIMARY_LABELS)
site_to_idx_map, n_sites_mapped = build_site_mapping(meta_full)
n_sites_cfg = CFG["proto_ssm"]["n_sites"]
site_ids_all, hours_all = get_file_metadata(meta_full, file_list, site_to_idx_map, n_sites_cfg)

# Per-file family labels
file_families = np.zeros((len(file_list), n_families), dtype=np.float32)
for fi in range(len(file_list)):
    active = np.where(labels_files[fi].sum(axis=0) > 0)[0]
    for ci in active:
        file_families[fi, class_to_family[ci]] = 1.0

# Create model
ssm_cfg = CFG["proto_ssm"]
model = ProtoSSMv2(
    d_input=emb_full.shape[1], d_model=ssm_cfg["d_model"], d_state=ssm_cfg["d_state"],
    n_ssm_layers=ssm_cfg["n_ssm_layers"], n_classes=N_CLASSES, n_windows=N_WINDOWS,
    dropout=ssm_cfg["dropout"], n_sites=ssm_cfg["n_sites"], meta_dim=ssm_cfg["meta_dim"],
).to(DEVICE)

model.init_prototypes_from_data(torch.tensor(emb_full, dtype=torch.float32), torch.tensor(Y_FULL, dtype=torch.float32))
model.init_family_head(n_families, class_to_family)

# v72: Load pretrained weights from 10K soundscape files
_pretrain_paths = [
    Path("/kaggle/input/birdclef2026-protossm-500ep/protossm_500ep_best.pt"),
    Path("/kaggle/input/protossm-10k/protossm_500ep_best.pt"),
]
_pretrain_loaded = False
for _pp in _pretrain_paths:
    if _pp.exists():
        print(f"Loading pretrained ProtoSSM from {_pp}")
        _ckpt = torch.load(str(_pp), map_location="cpu", weights_only=False)
        _model_dict = model.state_dict()
        _loaded, _skipped = 0, 0
        for k, v in _ckpt.items():
            if k in _model_dict and v.shape == _model_dict[k].shape:
                _model_dict[k] = v
                _loaded += 1
            else:
                _skipped += 1
        model.load_state_dict(_model_dict)
        model = model.to(DEVICE)
        print(f"  Loaded {_loaded} params, skipped {_skipped} (shape mismatch)")
        _pretrain_loaded = True
        break

if not _pretrain_loaded:
    print("No pretrained weights found -- training from scratch")

# Fine-tune on labeled data (always run, even with pretrained weights)
print(f"ProtoSSM v2 parameters: {model.count_parameters():,}")

t0 = time.time()
model, _ = train_proto_ssm_single(
    model, emb_files, logits_files, labels_files.astype(np.float32),
    site_ids_train=site_ids_all, hours_train=hours_all,
    file_families_train=file_families,
    cfg=CFG["proto_ssm_train"], verbose=True,
)
print(f"ProtoSSM training time: {time.time() - t0:.1f}s")

# OOF ensemble weight
ENSEMBLE_WEIGHT_PROTO = 0.55
print(f"Ensemble weight (ProtoSSM): {ENSEMBLE_WEIGHT_PROTO}")
print(f"Wall time after ProtoSSM: {(time.time() - _WALL_START) / 60:.1f} min")


In [ ]:
# Cell 15 -- v29/v40: ResidualSSM REMOVED (ablation experiment)
# v28 had ResidualSSM second-pass correction, v29 removes it to:
#   1. Save ~30s timeout margin
#   2. Reduce overfitting from second pass
#   3. Give ProtoSSM more training budget (80 vs 60 epochs)
res_model = None
CORRECTION_WEIGHT = 0.0
print("v29: ResidualSSM SKIPPED (ablation). ProtoSSM weight=0.55, epochs=80")
print(f"Wall time: {(time.time() - _WALL_START) / 60:.1f} min")


In [ ]:
# Cell 16 -- v48: Infer Perch on hidden test (batch_files=4, threads=1 for determinism)
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
if len(test_paths) == 0:
    print(f"Hidden test not mounted. Dry-run on first {CFG['dryrun_n_files']} train soundscapes.")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:CFG["dryrun_n_files"]]
else:
    print(f"Hidden test files: {len(test_paths)}")

# Wall-time guard
_elapsed = (time.time() - _WALL_START) / 60.0
print(f"Wall time before test inference: {_elapsed:.1f} min")

meta_test, scores_test_raw, emb_test = infer_perch_with_embeddings(
    test_paths, batch_files=CFG["batch_files"], verbose=True, proxy_reduce=CFG["proxy_reduce"])
print(f"Wall time after test inference: {(time.time()-_WALL_START)/60:.1f} min")
print(f"scores_test_raw: {scores_test_raw.shape}, emb_test: {emb_test.shape}")


In [ ]:
# Cell 16b -- EffNet-B0 5-fold inference on test soundscapes (v53: Perch + EffNet blend)
import timm
import librosa

_wall_effnet_start = time.time()

# -- EffNet audio processing (different from Perch) --
EFFNET_N_FFT = 1024
EFFNET_HOP = 512
EFFNET_MELS = 128
EFFNET_FMIN = 20
EFFNET_FMAX = 16000
EFFNET_SR = 32000
EFFNET_CLIP = 5.0
EFFNET_N_SAMPLES = int(EFFNET_SR * EFFNET_CLIP)

def compute_logmel_effnet(audio):
    """Log-mel spectrogram matching EffNet training exactly."""
    mel_spec = librosa.feature.melspectrogram(
        y=audio, sr=EFFNET_SR, n_fft=EFFNET_N_FFT, hop_length=EFFNET_HOP,
        n_mels=EFFNET_MELS, fmin=EFFNET_FMIN, fmax=EFFNET_FMAX, power=2.0)
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    vmin, vmax = log_mel.min(), log_mel.max()
    log_mel = (log_mel - vmin) / (vmax - vmin + 1e-8)
    return log_mel[np.newaxis, :, :].astype(np.float32)

# -- EffNet model (must match training architecture) --
class BirdCLEFEffNetLogits(nn.Module):
    def __init__(self, backbone_name='tf_efficientnet_b0_ns', pretrained=False,
                 num_species=234, dropout=0.3, in_chans=1):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, in_chans=in_chans)
        with torch.no_grad():
            dummy = torch.zeros(1, in_chans, 256, 256)
            emb_dim = self.backbone(dummy).shape[-1]
        self.head = nn.Sequential(nn.LayerNorm(emb_dim), nn.Dropout(dropout),
                                  nn.Linear(emb_dim, num_species))
    def forward(self, spec):
        if spec.shape[-2] != 256 or spec.shape[-1] != 256:
            spec = F.interpolate(spec, size=(256, 256), mode='bilinear', align_corners=False)
        return self.head(self.backbone(spec))

# -- Load EffNet checkpoints --
_effnet_dirs = [
    Path("/kaggle/input/datasets/mlnjsh/birdclef2026-effnet-5fold"),
    Path("/kaggle/input/birdclef2026-effnet-5fold"),
    Path("/kaggle/input/mlnjsh/birdclef2026-effnet-5fold"),
]
EFFNET_DIR = None
for d in _effnet_dirs:
    if d.exists():
        EFFNET_DIR = d
        break
if EFFNET_DIR is None:
    EFFNET_DIR = _effnet_dirs[0]  # fallback

effnet_models = []
if EFFNET_DIR.exists():
    ckpt_files = sorted(EFFNET_DIR.rglob('tf_efficientnet_b0_ns_fold*.pt'))
    if not ckpt_files:
        ckpt_files = sorted(EFFNET_DIR.rglob('*.pt'))
    for ckpt in ckpt_files[:5]:
        m = BirdCLEFEffNetLogits(pretrained=False)
        m.load_state_dict(torch.load(str(ckpt), map_location='cpu'))
        m.eval()
        effnet_models.append(m)
    print(f"Loaded {len(effnet_models)} EffNet checkpoints from {EFFNET_DIR}")
else:
    print(f"WARNING: EffNet dir not found: {EFFNET_DIR}")

# -- EffNet inference on test (no TTA to save time) --
effnet_logits = np.zeros((len(meta_test), N_CLASSES), dtype=np.float32)

if len(effnet_models) > 0:
    row_idx = 0
    for fi, fpath in enumerate(test_paths):
        try:
            audio, orig_sr = sf.read(str(fpath), dtype='float32', always_2d=False)
            if orig_sr != EFFNET_SR:
                audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=EFFNET_SR)
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
        except Exception as e:
            print(f"EffNet load error {fpath.name}: {e}")
            row_idx += N_WINDOWS
            continue

        for w in range(N_WINDOWS):
            s = w * EFFNET_N_SAMPLES
            e_idx = s + EFFNET_N_SAMPLES
            if e_idx <= len(audio):
                chunk = audio[s:e_idx]
            else:
                chunk = audio[s:]
                if len(chunk) < EFFNET_N_SAMPLES:
                    chunk = np.pad(chunk, (0, EFFNET_N_SAMPLES - len(chunk)))

            spec = compute_logmel_effnet(chunk)
            spec_t = torch.from_numpy(spec).unsqueeze(0)
            with torch.no_grad():
                preds = []
                for m in effnet_models:
                    logits = m(spec_t)
                    preds.append(torch.sigmoid(logits).numpy()[0])
                avg_prob = np.mean(preds, axis=0).astype(np.float32)
                # Convert to logits for blending
                avg_prob = np.clip(avg_prob, 1e-7, 1 - 1e-7)
                effnet_logits[row_idx] = np.log(avg_prob / (1 - avg_prob))
            row_idx += 1

        if (fi + 1) % 100 == 0:
            print(f"  EffNet [{fi+1}/{len(test_paths)}] {(time.time()-_wall_effnet_start)/60:.1f} min")

    print(f"EffNet inference done: {effnet_logits.shape}, {(time.time()-_wall_effnet_start)/60:.1f} min")
else:
    print("No EffNet models loaded -- skipping EffNet inference")
print(f"Wall time after EffNet: {(time.time()-_WALL_START)/60:.1f} min")


In [ ]:
# Cell 17 -- Score fusion: ProtoSSM v2 + MLP + ResidualSSM

# Step 1: ProtoSSM v2 inference on test
emb_test_files, test_file_list = reshape_to_files(emb_test, meta_test)
logits_test_files, _ = reshape_to_files(scores_test_raw, meta_test)
test_site_ids, test_hours = get_file_metadata(meta_test, test_file_list, site_to_idx_map, n_sites_cfg)

model.eval()
with torch.no_grad():
    proto_out, _, _ = model(
        torch.tensor(emb_test_files, dtype=torch.float32),
        torch.tensor(logits_test_files, dtype=torch.float32),
        site_ids=torch.tensor(test_site_ids, dtype=torch.long),
        hours=torch.tensor(test_hours, dtype=torch.long))
    proto_scores_flat = proto_out.numpy().reshape(-1, N_CLASSES).astype(np.float32)
print(f"ProtoSSM test scores: {proto_scores_flat.shape}")

# Step 2: Prior-fused base scores
test_base_scores, test_prior_scores = fuse_scores_with_tables(
    scores_test_raw, meta_test["site"].to_numpy(), meta_test["hour_utc"].to_numpy(), final_prior_tables)

# v27: Pseudo site priors (embedded as base64)
import base64, io
_PSEUDO_PRIORS_B64 = "..."
PSEUDO_PRIOR_ALPHA = 0.02
_buf = io.BytesIO(base64.b64decode(_PSEUDO_PRIORS_B64))
_d = np.load(_buf, allow_pickle=True)
pseudo_site_priors = _d["site_prior_matrix"]
pseudo_global_prior = _d["global_prior"]
pseudo_sites_list = list(_d["sites"])
if PSEUDO_PRIOR_ALPHA > 0:
    test_sites = meta_test["site"].to_numpy()
    site_to_pidx = {s: i for i, s in enumerate(pseudo_sites_list)}
    prior_matrix = np.tile(pseudo_global_prior, (len(test_sites), 1))
    for site, idx in site_to_pidx.items():
        mask = (test_sites == site)
        if mask.any():
            prior_matrix[mask] = pseudo_site_priors[idx]
    test_base_scores = (1 - PSEUDO_PRIOR_ALPHA) * test_base_scores + PSEUDO_PRIOR_ALPHA * prior_matrix
    print(f"Applied pseudo site priors (alpha={PSEUDO_PRIOR_ALPHA})")

# Step 3: MLP probe scores on test
emb_test_scaled = emb_scaler.transform(emb_test)
Z_TEST = emb_pca.transform(emb_test_scaled).astype(np.float32)
mlp_scores = test_base_scores.copy()

for cls_idx, clf in probe_models.items():
    X_cls_test = build_class_features(Z_TEST, raw_col=scores_test_raw[:, cls_idx],
                                       prior_col=test_prior_scores[:, cls_idx],
                                       base_col=test_base_scores[:, cls_idx])
    if hasattr(clf, "predict_proba"):
        prob = clf.predict_proba(X_cls_test)[:, 1].astype(np.float32)
        pred = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
    else:
        pred = clf.decision_function(X_cls_test).astype(np.float32)
    # v20: Per-class adaptive alpha
    n_pos = int(full_pos_counts[cls_idx])
    min_pos = int(BEST_PROBE["min_pos"])
    alpha_cls = 0.25 + 0.25 * min(1.0, (n_pos - min_pos) / 40.0)
    mlp_scores[:, cls_idx] = (1.0 - alpha_cls) * test_base_scores[:, cls_idx] + alpha_cls * pred


# Step 3b: Global MLP inference (v61: pre-trained on train_audio Perch embeddings)
GLOBAL_MLP_WEIGHT = CFG.get("global_mlp_weight", 0.0)
_gmlp_path = CFG.get("global_mlp_path")
if GLOBAL_MLP_WEIGHT > 0 and _gmlp_path is not None and _gmlp_path.exists():
    _gw = np.load(str(_gmlp_path))
    # 3-layer MLP: 1536 -> 768 -> 384 -> N_CLASSES (numpy forward pass, no PyTorch needed)
    _h = emb_test @ _gw['fc1_weight'].T + _gw['fc1_bias']
    _h = np.maximum(_h, 0)  # ReLU
    _h = _h @ _gw['fc2_weight'].T + _gw['fc2_bias']
    _h = np.maximum(_h, 0)  # ReLU
    global_mlp_logits = (_h @ _gw['fc3_weight'].T + _gw['fc3_bias']).astype(np.float32)
    del _gw, _h
    print(f"Global MLP logits: {global_mlp_logits.shape}, range [{global_mlp_logits.min():.3f}, {global_mlp_logits.max():.3f}]")
else:
    global_mlp_logits = None
    if GLOBAL_MLP_WEIGHT > 0:
        print(f"WARNING: Global MLP weights not found at {_gmlp_path}, skipping")
    else:
        print("Global MLP: DISABLED (weight=0)")

# Step 4: Ensemble (v61: Perch + EffNet + Global MLP)
perch_scores = (ENSEMBLE_WEIGHT_PROTO * proto_scores_flat + (1.0 - ENSEMBLE_WEIGHT_PROTO) * mlp_scores).astype(np.float32)
# v61: 3-way blend: Perch + EffNet + Global MLP
BLEND_WEIGHT_EFFNET = 0.08  # EffNet weight (proven in v53)
_blend_gmlp = GLOBAL_MLP_WEIGHT if global_mlp_logits is not None else 0.0
BLEND_WEIGHT_PERCH = 1.0 - BLEND_WEIGHT_EFFNET - _blend_gmlp
final_test_scores = BLEND_WEIGHT_PERCH * perch_scores
if effnet_logits is not None and np.abs(effnet_logits).sum() > 0:
    final_test_scores = final_test_scores + BLEND_WEIGHT_EFFNET * effnet_logits
    print(f"  + EffNet ({BLEND_WEIGHT_EFFNET:.0%})")
else:
    BLEND_WEIGHT_PERCH += BLEND_WEIGHT_EFFNET
    final_test_scores = BLEND_WEIGHT_PERCH * perch_scores
    print("  EffNet: SKIPPED (no scores)")
if global_mlp_logits is not None and _blend_gmlp > 0:
    final_test_scores = final_test_scores + _blend_gmlp * global_mlp_logits
    print(f"  + Global MLP ({_blend_gmlp:.0%})")
else:
    print("  Global MLP: SKIPPED")
final_test_scores = final_test_scores.astype(np.float32)
print(f"Blend weights: Perch={BLEND_WEIGHT_PERCH:.0%}, EffNet={BLEND_WEIGHT_EFFNET:.0%}, GlobalMLP={_blend_gmlp:.0%}")
print(f"Ensemble scores: {final_test_scores.shape}")

# Step 5: Residual correction
if res_model is not None and CORRECTION_WEIGHT > 0:
    first_pass_test_files, _ = reshape_to_files(final_test_scores, meta_test)
    res_model.eval()
    with torch.no_grad():
        test_corr = res_model(
            torch.tensor(emb_test_files, dtype=torch.float32),
            torch.tensor(first_pass_test_files, dtype=torch.float32),
            site_ids=torch.tensor(test_site_ids, dtype=torch.long),
            hours=torch.tensor(test_hours, dtype=torch.long)
        ).numpy().reshape(-1, N_CLASSES).astype(np.float32)
    final_test_scores = final_test_scores + CORRECTION_WEIGHT * test_corr
    print(f"Applied residual correction (weight={CORRECTION_WEIGHT})")
else:
    print("Residual correction: SKIPPED")

print(f"Score range: [{final_test_scores.min():.3f}, {final_test_scores.max():.3f}]")


In [ ]:
# Cell 18 -- Submission: per-taxon temperature + file-level confidence scaling
print(f"Wall time at submission: {(time.time()-_WALL_START)/60:.1f} min")

# v28: Per-taxon temperature scaling (from pantanal-distill)
temp_cfg = CFG["temperature"]
T_AVES = temp_cfg["aves"]
T_TEXTURE = temp_cfg["texture"]
class_temperatures = np.ones(N_CLASSES, dtype=np.float32) * T_AVES
for ci, label in enumerate(PRIMARY_LABELS):
    cn = CLASS_NAME_MAP.get(label, "Aves")
    if cn in TEXTURE_TAXA:
        class_temperatures[ci] = T_TEXTURE
print(f"Temperature: Aves={T_AVES}, Texture={T_TEXTURE}")

scaled_scores = final_test_scores / class_temperatures[None, :]
probs = sigmoid(scaled_scores)

# v28: File-level confidence scaling (from timeoptimized notebook)
top_k = CFG.get("file_level_top_k", 0)
if top_k > 0:
    print(f"Applying file-level confidence scaling (top_k={top_k})")
    probs = file_level_confidence_scale(probs, n_windows=N_WINDOWS, top_k=top_k)
    probs = np.clip(probs, 0.0, 1.0)

# Build submission
submission = pd.DataFrame(probs, columns=PRIMARY_LABELS)
submission.insert(0, "row_id", meta_test["row_id"].values)
submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].astype(np.float32)

expected_rows = len(test_paths) * N_WINDOWS
assert len(submission) == expected_rows, f"Expected {expected_rows}, got {len(submission)}"
assert submission.columns.tolist() == ["row_id"] + PRIMARY_LABELS
assert not submission.isna().any().any()
assert (submission[PRIMARY_LABELS] >= 0).all().all() and (submission[PRIMARY_LABELS] <= 1).all().all()

submission.to_csv("submission.csv", index=False)

print(f"Saved submission.csv -- shape: {submission.shape}")
print(f"Prob range: [{probs.min():.6f}, {probs.max():.6f}]")
print(f"Mean prob: {probs.mean():.4f}")
print(f"Total wall time: {(time.time()-_WALL_START)/60:.1f} min")
print(submission.iloc[:3, :8])


In [ ]:
# Cell 19 -- Final diagnostics
if MODE == "train":
    print("Modeled class count:", len(probe_models))
else:
    print("v28 submit mode completed.")
    print(f"Total wall time: {(time.time()-_WALL_START)/60:.1f} min")
